# Hospital Readmissions — Data Preparation Pipeline

> **Project:** BI + AI Analytics Portfolio  
> **Purpose:** Transform the raw Kaggle hospital readmissions dataset (17 columns) into a Tableau-ready dataset (33 columns) with engineered clinical dimensions, risk scores, and cost estimates.  
> **Output:** `hospital_readmissions_tableau_ready.csv`

---

## Notebook structure

| Section | Description |
|---|---|
| 1 | Import libraries |
| 2 | Load & inspect raw data |
| 3 | Data quality checks |
| 4 | Feature engineering (Steps 1–11) |
| 5 | Final column selection & export |
| 6 | Validation — readmission rates by key dimensions |

---
## Section 1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)

---
## Section 2 — Load & inspect raw data

In [ ]:
# Load the raw dataset
df = pd.read_csv('hospital_readmissions.csv')

print('Shape:', df.shape)
print('\nColumns:')
for col in df.columns:
    print(f'  {col}')

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Data types
df.dtypes

In [ ]:
# Numeric summary
df.describe()

---
## Section 3 — Data quality checks

In [ ]:
# Check for missing values
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print(f'\nTotal nulls: {df.isnull().sum().sum()}')

In [ ]:
# Check distinct values for all categorical columns
cat_cols = ['age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3',
            'glucose_test', 'A1Ctest', 'change', 'diabetes_med', 'readmitted']

for col in cat_cols:
    vals = df[col].value_counts()
    print(f'\n[{col}] — {df[col].nunique()} unique values')
    print(vals.to_string())

In [ ]:
# Baseline: overall readmission rate
overall_rate = (df['readmitted'] == 'yes').mean() * 100
readmit_count = (df['readmitted'] == 'yes').sum()

print(f'Total patients       : {len(df):,}')
print(f'Readmitted (yes)     : {readmit_count:,}')
print(f'Not readmitted (no)  : {len(df) - readmit_count:,}')
print(f'Overall readmission rate: {overall_rate:.1f}%')

---
## Section 4 — Feature Engineering

We engineer **16 new columns** across 11 steps, transforming 17 raw columns into 33 Tableau-ready columns.

### Step 1 — Age group standardisation

The raw `age` field uses bracket notation like `[60-70)`. We create three derived columns:
- `age_group` — clean readable label (e.g. `60–69`)
- `age_midpoint` — numeric midpoint for scatter plots and calculations
- `age_risk_tier` — clinical risk classification

In [ ]:
age_bucket_map = {
    '[40-50)': '40\u201349', '[50-60)': '50\u201359', '[60-70)': '60\u201369',
    '[70-80)': '70\u201379', '[80-90)': '80\u201389', '[90-100)': '90+'
}
age_midpoint_map = {
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65,
    '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
age_risk_map = {
    '[40-50)': 'Moderate', '[50-60)': 'Moderate', '[60-70)': 'High',
    '[70-80)': 'High',     '[80-90)': 'Very High', '[90-100)': 'Very High'
}

df['age_group']     = df['age'].map(age_bucket_map)
df['age_midpoint']  = df['age'].map(age_midpoint_map)
df['age_risk_tier'] = df['age'].map(age_risk_map)

print('age_group values:')
print(df['age_group'].value_counts().sort_index())

### Step 2 — Readmission binary flag & label

The raw `readmitted` column contains `yes`/`no`. We create:
- `readmitted_flag` — numeric 0/1 for aggregations and calculations in Tableau
- `readmitted_label` — capitalised string for chart labels

In [ ]:
df['readmitted_flag']  = (df['readmitted'] == 'yes').astype(int)
df['readmitted_label'] = df['readmitted'].str.capitalize()

print('Readmission distribution:')
print(df['readmitted_label'].value_counts())
print(f'\nReadmission rate: {df["readmitted_flag"].mean()*100:.1f}%')

### Step 3 — Length of stay bucket

Binning `time_in_hospital` into 4 clinical ranges for cleaner Tableau visualisation.

In [ ]:
bins   = [0, 2, 4, 7, 14]
labels = ['1\u20132 days', '3\u20134 days', '5\u20137 days', '8\u201314 days']

df['los_bucket'] = pd.cut(
    df['time_in_hospital'],
    bins=bins,
    labels=labels,
    right=True
)

print('LOS bucket distribution:')
print(df['los_bucket'].value_counts().sort_index())

### Step 4 — Clinical complexity score

A composite score (0–1) combining five clinical load indicators with weights:

| Indicator | Weight | Rationale |
|---|---|---|
| Medications | 30% | Polypharmacy is the strongest complexity signal |
| Lab procedures | 20% | Reflects diagnostic intensity |
| Procedures | 20% | Reflects interventional complexity |
| Prior inpatient | 15% | History of hospitalisation |
| Prior emergency | 15% | History of crisis events |

In [ ]:
df['complexity_score'] = (
    (df['n_medications']    / df['n_medications'].max()   ) * 0.30 +
    (df['n_lab_procedures'] / df['n_lab_procedures'].max()) * 0.20 +
    (df['n_procedures']     / df['n_procedures'].max()    ) * 0.20 +
    (df['n_inpatient']      / df['n_inpatient'].max()     ) * 0.15 +
    (df['n_emergency']      / df['n_emergency'].max()     ) * 0.15
).round(3)

complexity_labels = pd.cut(
    df['complexity_score'],
    bins=[0, 0.25, 0.50, 0.75, 1.01],
    labels=['Low', 'Moderate', 'High', 'Very High'],
    right=False
)
df['complexity_tier'] = complexity_labels

print('Complexity score stats:')
print(df['complexity_score'].describe())
print('\nComplexity tier distribution:')
print(df['complexity_tier'].value_counts())

### Step 5 — Prior utilisation score & tier

Combines all three prior visit types (outpatient, inpatient, emergency) into a single utilisation score, then bins into tiers.

In [ ]:
df['prior_utilisation'] = (
    df['n_outpatient'] + df['n_inpatient'] + df['n_emergency']
)

util_labels = pd.cut(
    df['prior_utilisation'],
    bins=[-1, 0, 2, 5, 200],
    labels=['None', 'Low', 'Moderate', 'High']
)
df['prior_utilisation_tier'] = util_labels

print('Prior utilisation distribution:')
print(df['prior_utilisation_tier'].value_counts())

print('\nReadmission rate by prior utilisation tier:')
print(
    df.groupby('prior_utilisation_tier')['readmitted_flag']
    .mean().mul(100).round(1)
)

### Step 6 — Medication burden tier

Bins `n_medications` into 4 clinical ranges to make medication load readable on charts.

In [ ]:
med_labels = pd.cut(
    df['n_medications'],
    bins=[0, 10, 20, 30, 80],
    labels=['Low (<10)', 'Moderate (10\u201320)', 'High (20\u201330)', 'Very High (30+)'],
    right=True
)
df['medication_burden'] = med_labels

print('Medication burden distribution:')
print(df['medication_burden'].value_counts())

### Step 7 — Primary diagnosis clean label

Maps the raw diagnosis codes/labels to clean, readable strings for Tableau.

In [ ]:
diag_readable = {
    'Circulatory':    'Circulatory Disease',
    'Respiratory':    'Respiratory Disease',
    'Digestive':      'Digestive Disease',
    'Diabetes':       'Diabetes',
    'Injury':         'Injury / Trauma',
    'Musculoskeletal':'Musculoskeletal',
    'Other':          'Other',
    'Missing':        'Unknown'
}
df['primary_diagnosis'] = df['diag_1'].map(diag_readable)

print('Primary diagnosis distribution:')
print(df['primary_diagnosis'].value_counts())

### Step 8 — Comorbidity count & flag

Counts how many of the three diagnosis fields (diag_1, diag_2, diag_3) are populated, and creates a binary comorbidity flag.

In [ ]:
df['comorbidity_count'] = (
    (df['diag_1'] != 'Missing').astype(int) +
    (df['diag_2'] != 'Missing').astype(int) +
    (df['diag_3'] != 'Missing').astype(int)
)
df['has_comorbidity'] = (
    df['comorbidity_count'] >= 2
).map({True: 'Yes', False: 'No'})

print('Comorbidity count distribution:')
print(df['comorbidity_count'].value_counts())
print('\nHas comorbidity:')
print(df['has_comorbidity'].value_counts())

### Step 9 — Diabetes management status

Classifies each patient's diabetes management quality into four categories based on medication use, medication changes, and test results.

| Status | Criteria |
|---|---|
| Not on Medication | `diabetes_med == 'no'` |
| Poorly Controlled | On medication + changed + high A1C or glucose |
| Stable | On medication + no medication change |
| Adjusting | On medication + changed + tests not alarming |

In [ ]:
def diabetes_status(row):
    on_med    = row['diabetes_med'] == 'yes'
    changed   = row['change'] == 'yes'
    a1c_high  = row['A1Ctest'] == 'high'
    gluc_high = row['glucose_test'] == 'high'

    if not on_med:
        return 'Not on Medication'
    if on_med and changed and (a1c_high or gluc_high):
        return 'Poorly Controlled'
    if on_med and not changed:
        return 'Stable'
    return 'Adjusting'

df['diabetes_mgmt_status'] = df.apply(diabetes_status, axis=1)

print('Diabetes management status distribution:')
print(df['diabetes_mgmt_status'].value_counts())
print('\nReadmission rate by diabetes management status:')
print(
    df.groupby('diabetes_mgmt_status')['readmitted_flag']
    .mean().mul(100).round(1).sort_values(ascending=False)
)

### Step 10 — Medical specialty clean label

Standardises the raw specialty strings into consistent clean labels.

In [ ]:
specialty_map = {
    'InternalMedicine':       'Internal Medicine',
    'Emergency/Trauma':       'Emergency / Trauma',
    'Family/GeneralPractice': 'Family / General Practice',
    'Cardiology':             'Cardiology',
    'Surgery':                'Surgery',
    'Other':                  'Other',
    'Missing':                'Not Recorded'
}
df['specialty_clean'] = df['medical_specialty'].map(specialty_map)

print('Specialty distribution:')
print(df['specialty_clean'].value_counts())

### Step 11 — Readmission risk score & tier

A composite risk score (0–8) built from six clinically meaningful binary indicators:

| Indicator | Points | Rationale |
|---|---|---|
| Any prior inpatient visit | +2 | Strongest predictor of readmission |
| Any prior emergency visit | +2 | High-acuity prior care |
| More than 15 medications | +1 | Polypharmacy risk |
| Age 70 or above | +1 | Geriatric risk factor |
| Has comorbidity | +1 | Multiple conditions |
| On diabetes medication | +1 | Chronic disease management |

In [ ]:
df['readmission_risk_score'] = (
    (df['n_inpatient']     > 0).astype(int) * 2 +
    (df['n_emergency']     > 0).astype(int) * 2 +
    (df['n_medications']   > 15).astype(int) * 1 +
    (df['age_midpoint']    >= 70).astype(int) * 1 +
    (df['has_comorbidity'] == 'Yes').astype(int) * 1 +
    (df['diabetes_med']    == 'yes').astype(int) * 1
)

risk_labels = pd.cut(
    df['readmission_risk_score'],
    bins=[-1, 1, 3, 5, 10],
    labels=['Low Risk', 'Moderate Risk', 'High Risk', 'Very High Risk']
)
df['readmission_risk_tier'] = risk_labels

print('Risk tier distribution:')
print(df['readmission_risk_tier'].value_counts())
print('\nReadmission rate by risk tier:')
print(
    df.groupby('readmission_risk_tier')['readmitted_flag']
    .mean().mul(100).round(1)
)

### Step 12 — Synthetic cost estimate

Since the dataset contains no cost field, we engineer a realistic cost estimate using a weighted formula based on clinical load indicators, with 12% random noise to simulate real-world variation.

> **Note:** This is a synthetic field for BI storytelling purposes only. It is not derived from actual billing data.

| Component | Rate | Basis |
|---|---|---|
| Length of stay | $1,800/day | US hospital avg daily rate |
| Lab procedures | $120 each | Avg diagnostic test cost |
| Procedures | $3,500 each | Avg interventional cost |
| Medications | $85 each | Avg medication cost |

In [ ]:
np.random.seed(42)  # For reproducibility

base_cost = (
    df['time_in_hospital'] * 1800 +
    df['n_lab_procedures'] * 120  +
    df['n_procedures']     * 3500 +
    df['n_medications']    * 85
)

# Add 12% random noise to simulate real-world cost variation
noise = np.random.normal(1.0, 0.12, len(df))
df['estimated_cost_usd'] = (base_cost * noise).round(0).astype(int)

print('Cost estimate stats:')
print(df['estimated_cost_usd'].describe())
print(f'\nTotal cohort cost: ${df["estimated_cost_usd"].sum():,.0f}')
print(f'Avg cost - readmitted:     ${df[df["readmitted_flag"]==1]["estimated_cost_usd"].mean():,.0f}')
print(f'Avg cost - not readmitted: ${df[df["readmitted_flag"]==0]["estimated_cost_usd"].mean():,.0f}')

### Step 13 — Add patient ID

Add a unique numeric patient identifier as the first column.

In [ ]:
df.insert(0, 'patient_id', range(1, len(df) + 1))
print(f'patient_id added: 1 to {len(df):,}')

---
## Section 5 — Final column selection & export

In [ ]:
# Select and order final columns for Tableau
final_cols = [
    'patient_id',
    # Demographics
    'age_group', 'age_midpoint', 'age_risk_tier',
    # Clinical
    'primary_diagnosis', 'diag_2', 'diag_3',
    'comorbidity_count', 'has_comorbidity',
    'specialty_clean',
    # Diabetes
    'diabetes_med', 'glucose_test', 'A1Ctest', 'change',
    'diabetes_mgmt_status',
    # Stay & procedures
    'time_in_hospital', 'los_bucket',
    'n_lab_procedures', 'n_procedures', 'n_medications',
    'medication_burden',
    # Prior utilisation
    'n_outpatient', 'n_inpatient', 'n_emergency',
    'prior_utilisation', 'prior_utilisation_tier',
    # Derived scores
    'complexity_score', 'complexity_tier',
    'readmission_risk_score', 'readmission_risk_tier',
    # Cost
    'estimated_cost_usd',
    # Target
    'readmitted_flag', 'readmitted_label'
]

df_final = df[final_cols].copy()

print(f'Final dataset shape: {df_final.shape}')
print(f'Original columns: 17  →  Final columns: {df_final.shape[1]}')
print('\nAll columns:')
for i, col in enumerate(df_final.columns, 1):
    print(f'  {i:>2}. {col}')

In [ ]:
# Preview final dataset
df_final.head()

In [ ]:
# Export to CSV
output_path = 'hospital_readmissions_tableau_ready.csv'
df_final.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Rows : {len(df_final):,}')
print(f'Cols : {len(df_final.columns)}')

---
## Section 6 — Validation

Verify the readmission rates match the expected values used in the Tableau dashboard.

In [ ]:
print('=== VALIDATION: READMISSION RATES ===')
print(f'\nOverall: {df_final["readmitted_flag"].mean()*100:.1f}%  (expected: 47.0%)')

print('\nBy primary diagnosis (expected: Diabetes = 53.6%):')
diag_rates = (
    df_final.groupby('primary_diagnosis')['readmitted_flag']
    .mean().mul(100).round(1).sort_values(ascending=False)
)
print(diag_rates)

print('\nBy age group:')
age_order = ['40\u201349','50\u201359','60\u201369','70\u201379','80\u201389','90+']
age_rates = (
    df_final.groupby('age_group')['readmitted_flag']
    .mean().mul(100).round(1)
)
print(age_rates.reindex(age_order))

print('\nBy risk tier (expected: Very High Risk = 63.5%):')
risk_rates = (
    df_final.groupby('readmission_risk_tier')['readmitted_flag']
    .agg(['mean','count'])
)
risk_rates['mean'] = risk_rates['mean'].mul(100).round(1)
risk_rates.columns = ['readmission_rate_%', 'patient_count']
print(risk_rates)

print('\nBy prior utilisation tier (expected: High = 78.6%):')
print(
    df_final.groupby('prior_utilisation_tier')['readmitted_flag']
    .mean().mul(100).round(1)
)

In [ ]:
print('=== FINAL SUMMARY ===')
print(f'Total patients          : {len(df_final):,}')
print(f'Total columns           : {len(df_final.columns)}')
print(f'Missing values          : {df_final.isnull().sum().sum()}')
print(f'Overall readmission rate: {df_final["readmitted_flag"].mean()*100:.1f}%')
print(f'Total estimated cost    : ${df_final["estimated_cost_usd"].sum():,.0f}')
print(f'\nOutput file ready for Tableau: hospital_readmissions_tableau_ready.csv')